In [ ]:
import json
from pathlib import Path


In [ ]:
basepath = "/Users/hmack/Development/smartrodent_experiments/runs"

detect_paths = ["detect001", "detect005", "detect01", "detect02"]

model_paths = [
    "yoloe_rodentornot",
    "yoloe_animalornot",
    "yolo26",
    "speciesnet",
    "biotrove-clip_species_names",
    "biotrove-clip_sub_df",
    "biotrove-clip_common_names",
    "biotrove-clip_animalornot",
]

species_classes = [
    "Rattus norvegicus",
    "Rattus rattus",
    "Mus musculus",
    "Myodes glareolus",
    "Apodemus agrarius",
    "Apodemus flavicollis",
    "Apodemus sylvaticus",
    "Microtus arvalis",
    "Microtus agrestis",
    "Crocidura leucodon",
]

common_name_classes = [
    "brown rat",
    "black rat",
    "house mouse",
    "bank vole",
    "striped field mouse",
    "yellow-necked mouse",
    "wood mouse",
    "common vole",
    "field vole",
    "bicolored white-toothed shrew",
]

animal_or_not = [
    "This is a photo of an animal like a mouse, rat, cat, fox, or hamster",
    "This is not a photo an animal at all, but a photo of something else entirely like trash, a car, a plastic box or a piece of wood or a leaf",
]

sub_df = [
    "This is a photo of a rodent like a rat, mouse or vole, or similar small mammal like a shrew",
    "This is a photo of not a rodent at all, but some other animal like a snake, bird or human ",
]

# when it has found these it has found the animal in the picture
yolo_animal_found = ["animal", "bear", "cat", "sheep", "cow", "dog", "bird"]

# when it reports these it has found the animal in the picture
species_net_animal_found = ["animal", "species", "family", "mammal"]

model_classes = {
    "yoloe_rodentornot": sub_df,
    "yoloe_animalornot": animal_or_not,
    "yolo26": yolo_animal_found,
    "speciesnet": species_net_animal_found,
    "biotrove-clip_species_names": species_classes,
    "biotrove-clip_sub_df": sub_df,
    "biotrove-clip_common_names": common_name_classes,
    "biotrove-clip_animalornot": animal_or_not,
}

In [ ]:
data = dict()
for dp in detect_paths:
    if dp not in data:
        data[dp] = dict()

    for mp in model_paths:
        if mp not in data[dp]:
            data[dp][mp] = dict()
        path = Path(basepath) / dp / mp / "central-europe"

        for sp in path.iterdir():
            if sp.name not in data[dp][mp]:
                data[dp][mp][sp.name] = dict()

            if ".DS_Store" in str(sp):
                continue
            results = json.loads((sp / "detections.json").read_text())

            for species, detects in results.items():
                for entry in detects:
                    cls = entry["class"]
                    conf = entry["conf"]

                    if mp == "yolo26":
                        if cls in [
                            "bird",
                            "bear",
                            "cat",
                            "dog",
                            "sheep",
                            "elephant",
                            "cow",
                            "giraffe",
                            "horse",
                            "zebra",
                        ]:
                            cls = "animal"
                        else:
                            cls = "nonanimal"

                    if cls not in data[dp][mp][sp.name]:
                        data[dp][mp][sp.name][cls] = dict()
                        data[dp][mp][sp.name][cls]["count"] = 1
                        data[dp][mp][sp.name][cls]["conf"] = [
                            conf,
                        ]
                    else:
                        data[dp][mp][sp.name][cls]["count"] += 1
                        data[dp][mp][sp.name][cls]["conf"].append(conf)

data

In [ ]:
import numpy as np
import pandas as pd

conf_summary_rows = []

for detect_path, models in data.items():
    for model_name, species_results in models.items():
        for species_name, class_results in species_results.items():
            for class_name, class_counts in class_results.items():
                if "conf" not in class_counts:
                    continue

                conf = np.asarray(class_counts["conf"], dtype=float)
                conf_summary_rows.append({
                    "detect_path": detect_path,
                    "model": model_name,
                    "species": species_name,
                    "class": class_name,
                    "count": conf.size,
                    "min": np.min(conf),
                    "q1": np.percentile(conf, 25),
                    "median": np.percentile(conf, 50),
                    "q3": np.percentile(conf, 75),
                    "max": np.max(conf),
                    "mean": np.mean(conf),
                })

conf_summary_df = pd.DataFrame(conf_summary_rows)
conf_summary_df

In [ ]:
conf_summary_df["class_canonical"] = (
       conf_summary_df["class"]
         .str.replace(r"^This is a photo of\s+", "", regex=True)
         .str.replace(r"^This is not a photo\s+", "", regex=True)
         .str.strip()
   )


conf_summary_df

In [ ]:
conf_summary_df.loc[:, "class_canonical"].unique()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

## Accuracy of rodent and animal identification with Yoloe and biotrove-clip

In [ ]:
subset = [
       "yoloe_rodentornot",
       "biotrove-clip_rodent_or_not",
   ]
sub_df = conf_summary_df[conf_summary_df["model"].isin(subset)]

group_cols = ['detect_path', 'model', 'species',]
sub_df["count_ratio"] = (
    sub_df["count"]
    / sub_df.groupby(group_cols)["count"].transform("sum")
)

g = sns.relplot(
    sub_df,
    x = "species",
    y="count_ratio",
    hue = "class_canonical",
    style="model",
    col="detect_path",
    col_wrap=2

)

for ax in g.axes.flatten():
    ax.tick_params(axis='x', labelrotation=90)

In [ ]:
subset = [
       "yoloe_animalornot",
       "biotrove-clip_animalornot",
   ]
sub_df = conf_summary_df[conf_summary_df["model"].isin(subset)]

group_cols = ['detect_path', 'model', 'species',]
sub_df["count_ratio"] = (
    sub_df["count"]
    / sub_df.groupby(group_cols)["count"].transform("sum")
)

g = sns.relplot(
    sub_df,
    x = "species",
    y="count_ratio",
    hue = "class_canonical",
    style="model",
    col="detect_path",
    col_wrap=2

)

for ax in g.axes.flatten():
    ax.tick_params(axis='x', labelrotation=90)

It seems that yoloe can reliably detect animals, but rodents not so much. Biotrove clip is largely useless

## Speciesnet animal identification accuracy
We might be able to identify animals with speciesnet, then train a yolo model on its bounding boxes (**check license first if this is permissible**), and then use that to get good animal crops but with a more accessible API. Yolo is far stronger in that regard than speciesnet. Alternatively, if speciesnet would work, we could fine tune it too. **Check if speciesnet can be retrained or finetuned**

In [ ]:
subset = [
       "speciesnet",
   ]
sub_df = conf_summary_df[conf_summary_df["model"].isin(subset)]

group_cols = ['detect_path', 'model', 'species',]
sub_df["count_ratio"] = (
    sub_df["count"]
    / sub_df.groupby(group_cols)["count"].transform("sum")
)

g = sns.relplot(
    sub_df,
    x = "species",
    y="count_ratio",
    hue = "class_canonical",
    style="model",
    col="detect_path",
    col_wrap=2

)

for ax in g.axes.flatten():
    ax.tick_params(axis='x', labelrotation=90)

## consolidate classes into 'animal' and 'not animal' classes 

In [ ]:
non_animal = [
    "vehicle",
    "blank",
    "human",
    "no cv result"
]
animal = [a for a in sub_df["class_canonical"].unique() if a not in non_animal]
animal, non_animal

#

In [ ]:
speciesnet_summary = conf_summary_df[conf_summary_df["model"].eq("speciesnet")].copy()
group_cols = ['detect_path', 'model', 'species',]



In [ ]:
speciesnet_summary["is_animal"] = np.where(
speciesnet_summary["class_canonical"].str.lower().str.strip().isin(animal),
       "animal",
       "no animal",
   )

group_cols = ["detect_path", "model", "species"]

speciesnet_animal_summary = (
    speciesnet_summary
    .groupby(group_cols + ["is_animal"], as_index=False, dropna=False)["count"]
    .sum()
)

speciesnet_animal_summary["count_ratio"] = (
    speciesnet_animal_summary["count"]
    / speciesnet_animal_summary.groupby(group_cols)["count"].transform("sum")
)


In [ ]:
speciesnet_animal_summary

In [ ]:
g = sns.relplot(
    speciesnet_animal_summary,
    x = "species",
    y="count_ratio",
    hue = "is_animal",
    style="detect_path",

)

for ax in g.axes.flatten():
    ax.tick_params(axis='x', labelrotation=90)

I was unsuccessful atm to change the confidence limit for speciesnet. however, it is rather confident in animal predictions, so extracting speciesnet crops could work well. Need to look it up better